In [1]:
import numpy as np
import pandas as pd
import numpy as np
import math
# import json
import os
import allensdk.core.swc as swc
# import psycopg2
from scipy.spatial.distance import euclidean
from morph_utils.templates import load_layer_template
from lims_utils import get_swc_from_lims
from query import query_lims_for_layers
from morph import to_dict, dict_to_Morphology
from fiducials import mouse_layer_edges, get_coords, convert_coords_str, upright_angle


In [2]:
csv =r'\\allen\programs\celltypes\workgroups\mousecelltypes\SarahWB\github_projects\primate_uprighting\testing\specimens.csv'
#raw/orginal file 
o_out = r'\\allen\programs\celltypes\workgroups\mousecelltypes\SarahWB\github_projects\primate_uprighting\testing\original'
upright_out = r'\\allen\programs\celltypes\workgroups\mousecelltypes\SarahWB\github_projects\primate_uprighting\testing\upright'

#make dirs if they don't exist 
if not os.path.isdir(upright_out): os.mkdir(upright_out)  
if not os.path.isdir(o_out): os.mkdir(o_out)

specimen_id_column = 'specimen_id'


In [3]:
# ##Check any conditions you need to HERE
df = pd.read_csv(csv)
df[specimen_id_column] = [a.replace(u'\u200b', '') for a in df[specimen_id_column].values]
print(len(df))
df.head()

20


,specimen_name,specimen_id,area,medial_side_to_viewerpov
0,Q21.26.013.12.03.01,1102613885,Pu,right
1,Q21.26.013.12.02.03​,1102623321,Pu,right
2,Q21.26.019.13.01.02,1110880399,Ca,right
3,Q21.26.019.13.01.05,1110896667,Ca,right
4,Q21.26.020.14.05.02,1112128140,ACB,left


In [4]:


avg_layer_depths = load_layer_template('mouse')['layers']
layer_edges = mouse_layer_edges(avg_layer_depths)
layer_list = ["1", "2/3",  "4", "5", "6a", "6b"]

In [5]:

def euclidean_distance(node1, node2):
    return euclidean(node1, node2)

def line(p1, p2):
    A = (p1[1] - p2[1])
    B = (p2[0] - p1[0])
    C = (p1[0]*p2[1] - p2[0]*p1[1])
    return A, B, -C

def intersection(L1, L2):
    D  = L1[0] * L2[1] - L1[1] * L2[0]
    Dx = L1[2] * L2[1] - L1[1] * L2[2]
    Dy = L1[0] * L2[2] - L1[2] * L2[0]
    if D != 0:
        x = Dx / D
        y = Dy / D
        return x,y
    else:
        return False
    
def find_translation(from_here, to_here):
    
    dx = to_here[0] - from_here[0]
    dy = to_here[1] - from_here[1]
    
    return dx, dy

def find_farthest(xs, ys, point):
    dist = 0
    opp = []
    for ix, x in enumerate(xs):
        y = ys[ix]
        this_dist = euclidean_distance(point, [x, y])
        if this_dist > dist:
            dist = this_dist
            opp = [x, y]
            
    return opp

def do_rotation(pts, angle):
    
    rot_x = []
    rot_y = []
    
    for p in pts:
#         print p
        rot = list(rotate([0,0], p, angle))
        rot_x.append(rot[0])
        rot_y.append(rot[1])
        
    return rot_x, rot_y

def unit_vector(vector):
    """ Returns the unit vector of the vector.  """
    return vector / np.linalg.norm(vector)

def angle_between(v1, v2):
    """ Returns the angle in radians between vectors 'v1' and 'v2'::

            >>> angle_between((1, 0, 0), (0, 1, 0))
            1.5707963267948966
            >>> angle_between((1, 0, 0), (1, 0, 0))
            0.0
            >>> angle_between((1, 0, 0), (-1, 0, 0))
            3.141592653589793
    """
    v1_u = unit_vector(v1)
    v2_u = unit_vector(v2)
    return np.arccos(np.clip(np.dot(v1_u, v2_u), -1.0, 1.0))


def rotate(origin, point, angle):
    """
    Rotate a point counterclockwise by a given angle around a given origin.

    The angle should be given in radians.
    """
    ox, oy = origin
    px, py = point

    qx = ox + math.cos(angle) * (px - ox) - math.sin(angle) * (py - oy)
    qy = oy + math.sin(angle) * (px - ox) + math.cos(angle) * (py - oy)
    return qx, qy


def determine_mirror():
    odx, ody = find_translation(lint, [0, 0])

    tp_x = plx + odx
    tp_y = ply + ody

    tm_x =  lx + odx
    tm_y = ly + ody

    opp = find_farthest(tp_x, tp_y, [0,0])

    
    opp.append(0)
    opp = tuple(opp)
    ##ccw angle
    ptheta= angle_between((1, 0, 0), opp)#(tp_x[0], tp_y[0], 0))

    rotp_x, rotp_y = do_rotation([[tp_x[0],tp_y[0]], [tp_x[-1], tp_y[-1]]], ptheta)
    rotm_x, rotm_y = do_rotation([[tm_x[0],tm_y[0]], [tm_x[-1], tm_y[-1]]], ptheta)

    opp = find_farthest(rotp_x, rotp_y, [0,0])
    opp.append(0)
    opp = tuple(opp)
    ctheta= angle_between((1, 0, 0), opp)#(tp_x[0], tp_y[0], 0))
    
    if ctheta != 0.0:
        
        rotp_x, rotp_y = do_rotation([[tp_x[0],tp_y[0]], [tp_x[-1], tp_y[-1]]], -ptheta)
        rotm_x, rotm_y = do_rotation([[tm_x[0],tm_y[0]], [tm_x[-1], tm_y[-1]]], -ptheta)

    mopp = find_farthest(rotm_x, rotm_y, [0,0])
    popp = find_farthest(rotp_x, rotp_y, [0,0])

    
    if mopp[1] < 0: #below the line
        if mopp[0] > popp[0]:
            return True
    else: #below line
        if mopp[0] < popp [0]:
            return True
        
    return False


In [6]:
uses_avg_layers = []

In [7]:

cells_with_issues = []
for ix, row in df.iterrows():
    
    specimen_id = row[specimen_id_column]
    oout = os.path.join(o_out, "{}.swc".format(specimen_id))
    uout = os.path.join(upright_out, "{}_upright.swc".format(specimen_id))

    try:
        
        print(ix, specimen_id)

        ldf = query_lims_for_layers(specimen_id)
        try:
            soma_coords, pia_coords, wm_coords, layer_coords = get_coords(ldf, layer_list)
        except:
            print("ERROR: Couldn't load layers for {}".format(specimen_id))
            cells_with_issues.append(specimen_id)
            continue

        try:
            swc_filename, swc_path = get_swc_from_lims(specimen_id)
        except TypeError:
            print("ERROR: Could not get swc for", specimen_id)
            cells_with_issues.append(specimen_id)
            continue
        swc_path = swc_path.replace('\\', '/')
        swc_path = swc_path.replace('/', '//', 1)
        morph = swc.read_swc(swc_path)
        morph.write(oout)

        if soma_coords is None:
            print("ERROR: No soma drawing for", specimen_id)
            cells_with_issues.append(specimen_id)
            continue


        #Edit WM coords
        row = ldf[ldf.draw_type == 'Pia']
        res = row.res.values[0]  
        pcoords = row.poly_coords.values[0]
        plx, ply = convert_coords_str(pcoords)
        plx = plx * res
        ply = ply * res
        L1 = line([plx[0],ply[0]], [plx[-1], ply[-1]])


        #Add White Matter
        row = ldf[ldf.draw_type == 'White Matter']
        res = row.res.values[0]  
        wcoords = row.poly_coords.values[0]
        lx, ly = convert_coords_str(wcoords)
        lx = lx * res
        ly = ly * res
        L2 = line([lx[0],ly[0]], [lx[-1], ly[-1]])

        lint = list(intersection(L1, L2))

        opp = find_farthest(lx, ly, lint)

        dx, dy = find_translation(lint, opp)
        new_lx = np.asarray(plx + dx)
        new_ly = np.asarray(ply + dy)

        wm_coords['x'] = pia_coords['x']
        wm_coords['y'] = pia_coords['y']

        pia_coords['x'] = new_lx
        pia_coords['y'] = new_ly


        theta, offset = upright_angle(layer_coords, soma_coords, pia_coords, wm_coords)
        theta += np.pi
        soma_node = morph.compartment_list_by_type(1)[0]
        aff = [1., 0., 0., 0., 1., 0., 0., 0., 1., -soma_node["x"], -soma_node["y"], -soma_node["z"]]
        morph.apply_affine(aff)
        aff = [np.cos(theta), -np.sin(theta), 0., np.sin(theta), np.cos(theta), 0., 0., 0., 1., 0., -offset, 0.]
        morph.apply_affine(aff)

        print("\tsaving uprighted morph {}".format(specimen_id))
        morph.save(uout)
        
        flip = determine_mirror()

        if flip:
            print("\tflipping morph {}".format(specimen_id))
            mdict = to_dict(uout)
            t = pd.DataFrame.from_dict(mdict).T
            t.x = t.x * -1
            tdict = t.to_dict(orient = 'index')
            tmorph = dict_to_Morphology(tdict)
            tmorph.save(uout)

        print

    except:
        print('issues with this cell...')
        cells_with_issues.append(specimen_id)
        continue

0 1102613885
	saving uprighted morph 1102613885
	flipping morph 1102613885
1 1102623321
	saving uprighted morph 1102623321
2 1110880399
	saving uprighted morph 1110880399
	flipping morph 1110880399
3 1110896667
	saving uprighted morph 1110896667
	flipping morph 1110896667
4 1112128140
	saving uprighted morph 1112128140
5 1112760246
	saving uprighted morph 1112760246
6 1112741472
	saving uprighted morph 1112741472
7 1096755367
	saving uprighted morph 1096755367
8 1096738774
	saving uprighted morph 1096738774
9 1112811000
	saving uprighted morph 1112811000
10 1096716775
	saving uprighted morph 1096716775
11 1096732440
	saving uprighted morph 1096732440
12 1079081411
ERROR: Couldn't load layers for 1079081411
13 1129691862
	saving uprighted morph 1129691862
	flipping morph 1129691862
14 1129687286
	saving uprighted morph 1129687286
	flipping morph 1129687286
15 1129673408
	saving uprighted morph 1129673408
	flipping morph 1129673408
16 1133704467
	saving uprighted morph 1133704467
	flippi

In [8]:
cells_with_issues


['1079081411']

In [9]:
#Use for troubleshooting
# for specimen_id in cells_with_issues:

#     oout = os.path.join(o_out, "{}.swc".format(specimen_id))
#     uout = os.path.join(upright_out, "{}_upright.swc".format(specimen_id))
    
        
#     print(specimen_id)

#     ldf = query_lims_for_layers(str(specimen_id))
#     try:
#         soma_coords, pia_coords, wm_coords, layer_coords = get_coords(ldf, layer_list)
#     except:
#         print("ERROR: Couldn't load layers for {:d}".format(specimen_id))
#         cells_with_issues.append(specimen_id)
#         continue

#     try:
#         swc_filename, swc_path = get_swc_from_lims(specimen_id)
# #         print swc_path
#     except TypeError:
#         print("ERROR: Could not get swc for", specimen_id)
#         cells_with_issues.append(specimen_id)
#         continue
#     swc_path = swc_path.replace('\\', '/')
#     swc_path = swc_path.replace('/', '//', 1)
#     morph = swc.read_swc(swc_path)
#     morph.write(oout)

#     if soma_coords is None:
#         print("ERROR: No soma drawing for", specimen_id)
#         cells_with_issues.append(specimen_id)
#         continue


#     #Edit WM coords
#     row = ldf[ldf.draw_type == 'Pia']
#     res = row.res.values[0]  
#     pcoords = row.poly_coords.values[0]
#     plx, ply = convert_coords_str(pcoords)
#     plx = plx * res
#     ply = ply * res
#     L1 = line([plx[0],ply[0]], [plx[-1], ply[-1]])


#     #Add White Matter
#     row = ldf[ldf.draw_type == 'White Matter']
#     res = row.res.values[0]  
#     wcoords = row.poly_coords.values[0]
#     lx, ly = convert_coords_str(wcoords)
#     lx = lx * res
#     ly = ly * res
#     L2 = line([lx[0],ly[0]], [lx[-1], ly[-1]])

#     lint = list(intersection(L1, L2))

#     opp = find_farthest(lx, ly, lint)

#     dx, dy = find_translation(lint, opp)
#     new_lx = np.asarray(plx + dx)
#     new_ly = np.asarray(ply + dy)

#     wm_coords['x'] = pia_coords['x']
#     wm_coords['y'] = pia_coords['y']

#     pia_coords['x'] = new_lx
#     pia_coords['y'] = new_ly


#     theta, offset = upright_angle(layer_coords, soma_coords, pia_coords, wm_coords)
#     theta += np.pi
#     soma_node = morph.compartment_list_by_type(1)[0]
#     aff = [1., 0., 0., 0., 1., 0., 0., 0., 1., -soma_node["x"], -soma_node["y"], -soma_node["z"]]
#     morph.apply_affine(aff)
#     aff = [np.cos(theta), -np.sin(theta), 0., np.sin(theta), np.cos(theta), 0., 0., 0., 1., 0., -offset, 0.]
#     morph.apply_affine(aff)

#     print("saving uprighted morph {}".format(specimen_id))
#     morph.save(uout)

#     flip = determine_mirror()

#     if flip:
#         print("flipping morph {}".format(specimen_id))
#         mdict = to_dict(uout)
#         t = pd.DataFrame.from_dict(mdict).T
#         t.x = t.x * -1
#         tdict = t.to_dict(orient = 'index')
#         tmorph = dict_to_Morphology(tdict)
#         tmorph.save(uout)

#     print

